# Mash Plus — ONE DAILY TRAINING NOTEBOOK (2026-10-24 → 2026-11-15)

**The same ConvGRU weights and AdamW optimizer continue across Kaggle versions.** This is not 23 unrelated trainings. A private Kaggle *dataset* caches only Hawizeh, Central Marshes and Sinnaf training imagery plus the latest research checkpoint; GitHub Actions republishes the one notebook each day, saves the finished checkpoint back to that dataset, and skips overlapping/failed versions.

**Scientific boundaries:** the already examined Hammar/Dalmaj regions are never loaded or evaluated here. Only chronological validation of the three training AOIs guides updates. Overall accuracy on dry pixels is not a sufficient criterion. A running model is not field-validated and daily repetitions of the same data are not independent evidence. The original Sentinel labels are MNDWI/NDVI satellite proxies; use independent human reference data and an untouched *new* site for competition claims.

The run intentionally uses **two epochs per day**, with weighted wet/transition sampling and bounded class loss. GitHub owns the private Kaggle API token; it is never embedded here. Source model is the real `MarshNetV3` ConvGRU, not a simulated predictor.

In [ ]:
# Run only with a Kaggle GPU; dependencies and code from documented open-source upstream.
%pip -q install "numpy>=1.26,<3"
import os,sys,json,subprocess,hashlib,zipfile,shutil
from pathlib import Path
import numpy as np
import torch
print("GPU",torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else "MISSING")
assert torch.cuda.is_available(),"GPU quota/accelerator unavailable: do not claim a completed GPU training day"
ROOT=Path("/kaggle/working/MarshPulse")
if not (ROOT/"ml/daily_train.py").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/ahpa5246-lgtm/MarshPulse.git",str(ROOT)],check=True)
COMMIT=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
assert len(COMMIT)==40
sys.path.insert(0,str(ROOT/"ml"))
from daily_bundle import extract_train_only,updated_after_training,validate_bundle,NAME
WORK=Path("/kaggle/working/mash_plus_daily")
WORK.mkdir(parents=True,exist_ok=True)
print("MODEL_SOURCE_COMMIT",COMMIT)

## Step 1 · Recover last real checkpoint without Kaggle credentials

The state is a **private Kaggle input dataset** versioned by a GitHub workflow only after the previous notebook has COMPLETED. The three training cube hashes and manifest consistency are checked on every resume. No hidden final-test data may appear in this bundle.

In [ ]:
state_files=list(Path("/kaggle/input").rglob(NAME))
if len(state_files)!=1:
    raise RuntimeError(f"Expected one private Mash Plus daily state input; got {len(state_files)}. Check GitHub dataset sync.")
BUNDLE=state_files[0]
manifest=validate_bundle(BUNDLE)
DATA=extract_train_only(BUNDLE,WORK/"train_only")
print("TRAIN_STATE",manifest["status"],"previous_completed_days",manifest.get("training_days",0))
assert not (DATA/"source_imagery/dalmaj_spatial_test").exists()
assert not (DATA/"source_imagery/hammar_spatial_test").exists()

## Step 2 · Continue actual PyTorch ConvGRU weights and optimizer

Validation/test data remain full-frame at natural wet prevalence. The same optimizer momentum and previous best validation weights are saved after training. The run is research-only even if validation scores improve.

In [ ]:
RESULT=WORK/"run"
state_in=DATA/"daily_continuation_state.pt"
seed=DATA/"v4_seed.pt"
cmd=[sys.executable,str(ROOT/"ml/daily_train.py"),"--data",str(DATA),
     "--out",str(RESULT),"--epochs","2","--batch","2","--lookback","4",
     "--base","16","--seed","17","--lr",".0001",
     "--source-commit",COMMIT,"--run-date",__import__("datetime").datetime.now(__import__("datetime").timezone.utc).date().isoformat()]
if state_in.is_file():cmd.extend(["--state-in",str(state_in)])
elif seed.is_file():cmd.extend(["--v4-seed",str(seed)])
print("CONTINUING_SAME_MODEL",state_in.is_file(),"WARM_START_V4",seed.is_file() and not state_in.is_file())
subprocess.run(cmd,check=True)
REPORT=json.loads((RESULT/"daily_training_report.json").read_text())
assert REPORT["status"]=="RESEARCH_ONLY_NOT_FIELD_VALIDATED"
print("DAILY_RESULT",json.dumps(REPORT,indent=2))

## Step 3 · Export exactly one verifiable state ZIP for next day's notebook

Only complete days export. This includes the three source images, unchanged manifests, current model weights, best validation-only weights, optimizer state and a plaintext research report. The output ZIP is consumed **tomorrow** by GitHub's authenticated bridge; GPU quotas and scheduling may prevent a particular day's execution.

In [ ]:
NEXT=Path("/kaggle/working")/NAME
updated_after_training(BUNDLE,RESULT,NEXT)
assert NEXT.is_file() and NEXT.stat().st_size>0
print("DAILY_STATE_READY",NEXT,"BYTES",NEXT.stat().st_size,
      "SHA256",hashlib.sha256(NEXT.read_bytes()).hexdigest())
print("FIELD_VALIDATION FALSE; FINAL_GEOGRAPHIC_TEST NOT_USED; NO_SCIENTIFIC_CERTIFICATION")